In [1]:
# check_models.ipynb
# Short notebook to check weights + requirements for all ONNX models.

# Cell 1 — install deps (run once)
!pip install onnx onnxruntime numpy

# Cell 2 — imports + model list
import os
import onnx
import onnxruntime as ort
import numpy as np

MODEL_PATHS = [
    "best.onnx",
    "eartags_horn.onnx",
    "head_left_right.onnx",
    "left_right_flank_dead.onnx",
    "left_right_live.onnx",
    "live_cattle_front_head.onnx",
    "yolov8n.onnx",
]

def human(nbytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if nbytes < 1024:
            return f"{nbytes:.1f} {unit}"
        nbytes /= 1024
    return f"{nbytes:.1f} TB"

# Cell 3 — inspect each model: size, inputs, outputs, opset, params
for path in MODEL_PATHS:
    print("=" * 70)
    print(f"MODEL: {path}")
    print("=" * 70)

    if not os.path.exists(path):
        print("  ❌ File not found\n")
        continue

    # --- file size on disk ---
    size = os.path.getsize(path)
    print(f"  File size:        {human(size)} ({size:,} bytes)")

    # --- ONNX graph metadata ---
    try:
        model = onnx.load(path, load_external_data=False)
        graph = model.graph

        print(f"  IR version:       {model.ir_version}")
        print(f"  Opset:            {[(o.domain or 'ai.onnx', o.version) for o in model.opset_import]}")
        print(f"  Producer:         {model.producer_name} {model.producer_version}".rstrip())

        # --- inputs ---
        print("  Inputs:")
        for inp in graph.input:
            dims = [d.dim_value if d.dim_value > 0 else (d.dim_param or "?")
                    for d in inp.type.tensor_type.shape.dim]
            print(f"    - {inp.name}: {dims}")

        # --- outputs ---
        print("  Outputs:")
        for out in graph.output:
            dims = [d.dim_value if d.dim_value > 0 else (d.dim_param or "?")
                    for d in out.type.tensor_type.shape.dim]
            print(f"    - {out.name}: {dims}")

        # --- parameter count ---
        total_params = 0
        for init in graph.initializer:
            n = 1
            for d in init.dims:
                n *= d
            total_params += n
        print(f"  Parameters:       {total_params:,}  (~{total_params/1e6:.2f} M)")

    except Exception as e:
        print(f"  ⚠️  onnx.load failed: {e}")

    # --- actual inference session (real runtime requirements) ---
    try:
        sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
        print("  Inference OK:")
        print(f"    Providers:      {sess.get_providers()}")
        for i in sess.get_inputs():
            print(f"    Input  '{i.name}': shape={i.shape}, type={i.type}")
        for o in sess.get_outputs():
            print(f"    Output '{o.name}': shape={o.shape}, type={o.type}")
    except Exception as e:
        print(f"  ⚠️  InferenceSession failed: {e}")

    print()

# Cell 4 — environment / requirement summary
print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)
import sys
print(f"  Python:           {sys.version.split()[0]}")
print(f"  onnx:             {onnx.__version__}")
print(f"  onnxruntime:      {ort.__version__}")
print(f"  numpy:            {np.__version__}")
print(f"  ORT providers:    {ort.get_available_providers()}")

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albumentations 2.0.8 requires pydantic>=2.9.2, but you have pydantic 2.5.0 which is incompatible.
googleapis-common-protos 1.72.0 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.4.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 7.36.2 which is incompatible.
google-api-core 2.29.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.19.5, but you have protobuf 7.36.2 which is incompatible.
google-cloud-vision 3.11.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.2

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   --------- ------------------------------ 1.8/7.9 MB 10.1 MB/s eta 0:00:01
   ---------------------- ----------------- 4.5/7.9 MB 10.8 MB/s eta 0:00:01
   ------------------------------- -------- 6.3/7.9 MB 10.4 MB/s eta 0:00:01
   ---------------------------------------- 7.9/7.9 MB 10.2 MB/s  0:00:00
   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/14.3 MB 10.7 MB/s eta 0:00:02
   ------------ --------------------------- 4.5/14.3 MB 11.2 MB/s eta 0:00:01
   ------------------ --------------------- 6.6/14.3 MB 10.9 MB/s eta 0:00:01
   --------------------- ------------------ 7.9/14.3 MB 9.7 MB/s eta 0:00:01
   ---------------------------- ----------- 10.2/14.3 MB 10.1 MB/s eta 0:00:01
   -----------------